In [4]:
import pandas as pd

# Load all tables
train    = pd.read_csv(r"D:\PROJECTS\customer-churn-prediction\train_v2.csv\data\churn_comp_refresh\train_v2.csv")
members  = pd.read_csv(r"D:\PROJECTS\customer-churn-prediction\members_v3.csv\members_v3.csv")
txn_agg  = pd.read_csv(r"D:\PROJECTS\customer-churn-prediction\txn_agg.csv")
log_agg  = pd.read_csv(r"D:\PROJECTS\customer-churn-prediction\log_agg.csv")

print("Loaded all tables")
print(f"train: {train.shape}, members: {members.shape}, txn_agg: {txn_agg.shape}, log_agg: {log_agg.shape}")

Loaded all tables
train: (970960, 2), members: (6769473, 6), txn_agg: (1197050, 8), log_agg: (1103894, 7)


In [5]:
# Clean age — bd has outliers, 0 means missing
members['age'] = members['bd'].where(members['bd'].between(10, 80), other=None)
members = members.drop(columns=['bd'])

# Extract registration year
members['registration_year'] = pd.to_datetime(
    members['registration_init_time'], format='%Y%m%d', errors='coerce'
).dt.year
members = members.drop(columns=['registration_init_time'])

print(members[['age', 'registration_year']].describe())

                age  registration_year
count  2.218915e+06       6.769473e+06
mean   2.939145e+01       2.014451e+03
std    1.021897e+01       2.323070e+00
min    1.000000e+01       2.004000e+03
25%    2.200000e+01       2.014000e+03
50%    2.700000e+01       2.015000e+03
75%    3.500000e+01       2.016000e+03
max    8.000000e+01       2.017000e+03


In [6]:
# Merge all on msno using left joins
df = train.merge(members, on='msno', how='left')
df = df.merge(txn_agg,   on='msno', how='left')
df = df.merge(log_agg,   on='msno', how='left')

print(f"Merged shape: {df.shape}")
print(df.isnull().sum())

Merged shape: (970960, 20)
msno                      0
is_churn                  0
city                 109993
gender               582055
registered_via       109993
age                  584567
registration_year    109993
last_price            37382
last_paid             37382
last_auto_renew       37382
last_cancel           37382
total_cancels         37382
num_transactions      37382
last_expire           37382
total_secs           216409
songs_100            216409
songs_uniq           216409
active_days          216409
avg_secs_per_day     216409
completion_rate      216409
dtype: int64


In [7]:
# One-hot encode categoricals
df = pd.get_dummies(df, columns=['city', 'gender', 'registered_via'], dummy_na=True)

# Drop ID and fill nulls
df = df.drop(columns=['msno'])
df = df.fillna(0)

print(f"Final shape: {df.shape}")
df.to_csv(r"D:\PROJECTS\customer-churn-prediction\merged_features.csv", index=False)
print("Done! merged_features.csv saved.")

Final shape: (970960, 47)
Done! merged_features.csv saved.
